In [1]:
import os

REPO = 'giomamaca/walmart-sales-forecasting'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    from google.colab import userdata, files

    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
        print('Upload your kaggle.json')
        uploaded = files.upload()
        shutil.move('kaggle.json', f'{kaggle_dir}/kaggle.json')
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)

    github_token = userdata.get('GITHUB_TOKEN')
    repo_dir = f'/content/{REPO.split("/")[1]}'
    if not os.path.exists(repo_dir):
        !git clone -q https://{github_token}@github.com/{REPO}.git {repo_dir}
    os.chdir(repo_dir)

    !pip install -q kaggle mlflow wandb lightgbm

    import wandb
    wandb.login(key=userdata.get('WANDB_API_KEY'))

    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
    !unzip -o -q data/walmart-recruiting-store-sales-forecasting.zip -d data
    !for f in data/*.zip; do unzip -o -q "$f" -d data; done

    !mlflow db upgrade sqlite:///mlflow.db

    DATA_DIR = 'data'
else:
    DATA_DIR = '..'

Upload your kaggle.json


Saving kaggle.json to kaggle.json
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adola23 (adola23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100% 2.70M/2.70M [00:00<00:00, 178MB/s]

2026/07/09 16:01:14 INFO mlflow.store.db.utils: Updating database tables


In [2]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import wandb
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from lightgbm import LGBMRegressor

In [3]:
train = pd.read_csv(f'{DATA_DIR}/train.csv', parse_dates=['Date'])
features = pd.read_csv(f'{DATA_DIR}/features.csv', parse_dates=['Date'])
stores = pd.read_csv(f'{DATA_DIR}/stores.csv')

train.shape, features.shape, stores.shape

((421570, 5), (8190, 12), (45, 3))

In [4]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.average(np.abs(y_true - y_pred), weights=weights)

In [5]:
MARKDOWN_COLS = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
PRE_CHRISTMAS = pd.to_datetime(['2010-12-24', '2011-12-23', '2012-12-21'])

FEATURE_COLS = [
    'Store', 'Dept', 'IsHoliday', 'Size', 'Type',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
    'Year', 'Month', 'Week', 'IsPreChristmas',
    'Sales_Lag52', 'Sales_Lag52_Mean3',
    'Series_Mean', 'Series_Median', 'Series_Std', 'Series_WoyMean',
    'Dept_Mean', 'Store_Mean',
]

# v2: v1-is featurebs emateba istoriuli feature-ebi. yvela lag >= 51 kviraa,
# amitom test-is nebismieri tarigistvis mnishvneloba train-is istoriidan modis
# da rekursiuli prognozi ar gvchirdeba.
class FeatureBuilderV2(BaseEstimator, TransformerMixin):
    def __init__(self, stores_df, features_df):
        self.stores_df = stores_df
        self.features_df = features_df

    def fit(self, X, y=None):
        df = X[['Store', 'Dept', 'Date']].copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df['y'] = y.values if hasattr(y, 'values') else y

        self.history_ = df.set_index(['Store', 'Dept', 'Date'])['y']
        self.series_stats_ = df.groupby(['Store', 'Dept'])['y'].agg(['mean', 'median', 'std'])
        woy = df['Date'].dt.isocalendar().week.astype(int)
        self.woy_mean_ = df.assign(woy=woy).groupby(['Store', 'Dept', 'woy'])['y'].mean()
        self.dept_mean_ = df.groupby('Dept')['y'].mean()
        self.store_mean_ = df.groupby('Store')['y'].mean()
        return self

    def _lag(self, df, weeks):
        idx = pd.MultiIndex.from_arrays(
            [df['Store'], df['Dept'], df['Date'] - pd.Timedelta(weeks=weeks)])
        return self.history_.reindex(idx).values

    def transform(self, X):
        df = X.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.merge(self.stores_df, on='Store', how='left')
        df = df.merge(self.features_df.drop(columns=['IsHoliday']), on=['Store', 'Date'], how='left')

        df[MARKDOWN_COLS] = df[MARKDOWN_COLS].fillna(0)
        df['Type'] = df['Type'].map({'A': 0, 'B': 1, 'C': 2})
        df['IsHoliday'] = df['IsHoliday'].astype(int)

        df['Year'] = df['Date'].dt.year
        df['Month'] = df['Date'].dt.month
        df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
        df['IsPreChristmas'] = df['Date'].isin(PRE_CHRISTMAS).astype(int)

        df['Sales_Lag52'] = self._lag(df, 52)
        lags = pd.DataFrame({w: self._lag(df, w) for w in (51, 52, 53)})
        df['Sales_Lag52_Mean3'] = lags.mean(axis=1).values

        pair_idx = pd.MultiIndex.from_arrays([df['Store'], df['Dept']])
        stats = self.series_stats_.reindex(pair_idx)
        df['Series_Mean'] = stats['mean'].values
        df['Series_Median'] = stats['median'].values
        df['Series_Std'] = stats['std'].values

        woy_idx = pd.MultiIndex.from_arrays(
            [df['Store'], df['Dept'], df['Date'].dt.isocalendar().week.astype(int)])
        df['Series_WoyMean'] = self.woy_mean_.reindex(woy_idx).values
        df['Dept_Mean'] = self.dept_mean_.reindex(df['Dept']).values
        df['Store_Mean'] = self.store_mean_.reindex(df['Store']).values

        return df[FEATURE_COLS]

In [6]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('LightGBM_v2_Training')

2026/07/09 16:01:29 INFO mlflow.tracking.fluent: Experiment with name 'LightGBM_v2_Training' does not exist. Creating a new experiment.


<Experiment: artifact_location='/content/walmart-sales-forecasting/mlruns/5', creation_time=1783612889628, effective_trace_archival_retention=None, experiment_id='5', last_update_time=1783612889628, lifecycle_stage='active', name='LightGBM_v2_Training', tags={}, trace_location=None, workspace='default'>

In [7]:
with mlflow.start_run(run_name='LightGBM_v2_Cleaning'):
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_v2_Cleaning', reinit='finish_previous')

    n_negative = int((train['Weekly_Sales'] < 0).sum())
    missing_markdown_pct = float(features[MARKDOWN_COLS].isna().mean().mean())

    mlflow.log_param('negative_sales_rows', n_negative)
    mlflow.log_param('negative_sales_action', 'clip_to_zero')
    mlflow.log_metric('missing_markdown_pct', missing_markdown_pct)
    wandb.log({'negative_sales_rows': n_negative, 'missing_markdown_pct': missing_markdown_pct})
    wandb.finish()

    y_train = train['Weekly_Sales'].clip(lower=0)

missing_markdown_pct,▁
negative_sales_rows,▁
missing_markdown_pct,0.55849
negative_sales_rows,1285


In [8]:
with mlflow.start_run(run_name='LightGBM_v2_Feature_Selection'):
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_v2_Feature_Selection', reinit='finish_previous')

    builder = FeatureBuilderV2(stores, features)
    builder.fit(train[['Store', 'Dept', 'Date']], y_train)
    X_all = builder.transform(train)

    probe = LGBMRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, verbose=-1)
    probe.fit(X_all, y_train)

    importances = pd.Series(probe.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
    mlflow.log_dict(importances.to_dict(), 'feature_importances.json')
    wandb.log({'feature_importances': wandb.Table(dataframe=importances.reset_index().rename(columns={'index': 'feature', 0: 'importance'}))})

    selected_features = importances[importances > 0].index.tolist()
    mlflow.log_param('n_features_selected', len(selected_features))
    mlflow.log_dict({'selected_features': selected_features}, 'selected_features.json')
    wandb.log({'n_features_selected': len(selected_features)})
    wandb.finish()

importances

n_features_selected,▁
n_features_selected,26


,0
Series_WoyMean,1033
Sales_Lag52,584
Series_Std,572
CPI,384
Week,383
Dept,300
Unemployment,296
Dept_Mean,276
Series_Mean,215
Fuel_Price,212


In [9]:
def time_based_splits(dates, n_splits=3):
    dates = np.sort(dates.unique())
    fold_size = len(dates) // (n_splits + 1)
    splits = []
    for i in range(1, n_splits + 1):
        train_end = dates[fold_size * i - 1]
        val_end = dates[min(fold_size * (i + 1) - 1, len(dates) - 1)]
        splits.append((train_end, val_end))
    return splits

splits = time_based_splits(train['Date'])
splits

[(np.datetime64('2010-10-01T00:00:00.000000000'),
  np.datetime64('2011-06-03T00:00:00.000000000')),
 (np.datetime64('2011-06-03T00:00:00.000000000'),
  np.datetime64('2012-02-03T00:00:00.000000000')),
 (np.datetime64('2012-02-03T00:00:00.000000000'),
  np.datetime64('2012-10-05T00:00:00.000000000'))]

In [10]:
params = dict(n_estimators=1500, num_leaves=255, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, verbose=-1)

with mlflow.start_run(run_name='LightGBM_v2_CV'):
    mlflow.log_params(params)
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_v2_CV', config=params, reinit='finish_previous')

    fold_scores = []
    for i, (train_end, val_end) in enumerate(splits):
        train_mask = train['Date'] <= train_end
        val_mask = (train['Date'] > train_end) & (train['Date'] <= val_end)

        pipe = Pipeline([
            ('features', FeatureBuilderV2(stores, features)),
            ('model', LGBMRegressor(**params)),
        ])
        pipe.fit(train.loc[train_mask, ['Store', 'Dept', 'Date', 'IsHoliday']], y_train[train_mask])
        preds = pipe.predict(train.loc[val_mask, ['Store', 'Dept', 'Date', 'IsHoliday']])

        score = wmae(y_train[val_mask].values, preds, train.loc[val_mask, 'IsHoliday'].values)
        fold_scores.append(score)
        mlflow.log_metric('wmae', score, step=i)
        wandb.log({'fold': i, 'wmae': score})

    mlflow.log_metric('wmae_mean', float(np.mean(fold_scores)))
    mlflow.log_metric('wmae_std', float(np.std(fold_scores)))
    wandb.log({'wmae_mean': float(np.mean(fold_scores)), 'wmae_std': float(np.std(fold_scores))})
    wandb.finish()

fold_scores

fold,▁▅█
wmae,█▁▁
wmae_mean,▁
wmae_std,▁
fold,2
wmae,2224.23241
wmae_mean,4492.01494
wmae_std,3378.71991


[np.float64(9268.226086915653),
 np.float64(1983.5863146517559),
 np.float64(2224.232408674201)]

In [11]:
with mlflow.start_run(run_name='LightGBM_v2_Final'):
    mlflow.log_params(params)
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_v2_Final', config=params, reinit='finish_previous')

    pipeline = Pipeline([
        ('features', FeatureBuilderV2(stores, features)),
        ('model', LGBMRegressor(**params)),
    ])
    pipeline.fit(train[['Store', 'Dept', 'Date', 'IsHoliday']], y_train)

    preds = pipeline.predict(train[['Store', 'Dept', 'Date', 'IsHoliday']])
    train_wmae = wmae(y_train.values, preds, train['IsHoliday'].values)
    mlflow.log_metric('train_wmae', train_wmae)
    wandb.log({'train_wmae': train_wmae})

    mlflow.sklearn.log_model(pipeline, name='model', serialization_format='cloudpickle')
    wandb.finish()

train_wmae

2026/07/09 16:09:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


train_wmae,▁
train_wmae,551.89666


np.float64(551.896655239971)

In [12]:
if IN_COLAB:
    import requests
    gh_user = requests.get('https://api.github.com/user', headers={'Authorization': f'token {github_token}'}).json()['login']
    !git config user.name "{gh_user}"
    !git config user.email "{gh_user}@users.noreply.github.com"

    !git add mlflow.db mlruns
    !git commit -m "Run model_experiment_LightGBM_v2.ipynb in Colab"
    !git push

[main 2dae9f9] Run model_experiment_LightGBM_v2.ipynb in Colab
 8 files changed, 133 insertions(+)
 create mode 100644 mlruns/5/d60db6380e6549379fd1c8d673070287/artifacts/feature_importances.json
 create mode 100644 mlruns/5/d60db6380e6549379fd1c8d673070287/artifacts/selected_features.json
 create mode 100644 mlruns/5/models/m-ac3e9216501d44bba7f78087836e2a45/artifacts/MLmodel
 create mode 100644 mlruns/5/models/m-ac3e9216501d44bba7f78087836e2a45/artifacts/conda.yaml
 create mode 100644 mlruns/5/models/m-ac3e9216501d44bba7f78087836e2a45/artifacts/model.pkl
 create mode 100644 mlruns/5/models/m-ac3e9216501d44bba7f78087836e2a45/artifacts/python_env.yaml
 create mode 100644 mlruns/5/models/m-ac3e9216501d44bba7f78087836e2a45/artifacts/requirements.txt
Enumerating objects: 20, done.
Counting objects: 100% (20/20), done.
Delta compression using up to 2 threads
Compressing objects: 100% (15/15), done.
Writing objects: 100% (17/17), 14.40 MiB | 3.18 MiB/s, done.
Total 17 (delta 3), reused 3 (d